# Finite-Difference Strategies for Gradient Checking and AD-via-FD

Benchmark for [#749](https://github.com/pasteurlabs/tesseract-core/issues/749).

This is a draft to validate the structure of the demo. 

Solvers:

- **Rosenbrock** ([`examples/univariate`](https://github.com/pasteurlabs/tesseract-core/tree/main/examples/univariate)), rescaled so its inputs differ by four orders of magnitude ([#706](https://github.com/pasteurlabs/tesseract-core/issues/706)), with optional noise. Exact gradient known.
- **Heat transfer, JAX-FEM** ([mosaic](https://github.com/pasteurlabs/mosaic)). A real solver with a correct VJP. We trust this gradient because independent implementations agree with it: torch-fem, and the corrected FEniCS adjoint (to 7e-9).
- **Heat transfer, FEniCS**, at the version whose adjoint was 5% off ([mosaic#158](https://github.com/pasteurlabs/mosaic/issues/158)). JAX-FEM gives its reference gradient.

Would our checks have caught that 5% bug?

## Chapter 1: Why $\varepsilon$ matters

Tesseract's finite differences. Absolute step `eps`, and call `apply` twice:

$$D_h f(x; v) = \frac{f(x + h v) - f(x - h v)}{2h} \approx J v$$

Describe : 
- `check_gradients` compares it with the endpoint's value for $v = e_i$, 
- `finite_difference_jvp` evaluates it along the tangent directly. 
- `finite_difference_vjp` assembles $J$ one column at a time, at a cost of $2n$ calls to `apply`.


### 1.1 V-curves

No single `eps` is best for every input (Rosenbrock, then JAX-FEM). A failure can reflect a badly chosen eps rather than a wrong gradient (#706)

Code: sweep $h$ from $10^{-12}$ to $1$, per input and noise level $\sigma$, computing the relative error of $D_h f$ against the reference gradient. On Rosenbrock the noise is injected (float64, deterministic in $x$) and compared with the exact gradient; on JAX-FEM it comes from the solver tolerance.

Output: error against $h$ on log-log axes, one panel per input, one line per $\sigma$, with slope-2 and slope-1 guides and the default `eps=1e-4` marked.

## Chapter 2: FD for gradient estimation

### 2.1 Ways to choose the step

- **Fixed `eps`**: the current default, `1e-4`.
- **Per-input `eps`**: supported by the FD helpers ([#712](https://github.com/pasteurlabs/tesseract-core/pull/712)) and `check-gradients` ([#713](https://github.com/pasteurlabs/tesseract-core/pull/713)). SPSA and randomized estimators ? 
- **Scaled to the input**: proposed in [#516](https://github.com/pasteurlabs/tesseract-core/issues/516).
- **Best of a sweep**: as mosaic does, proposed in [#740](https://github.com/pasteurlabs/tesseract-core/issues/740).

Code: each method returns `eps` as a mapping `{path: h_i}`, usable by both the FD helpers and `check_gradients`.

Output: the chosen steps as markers on the V-curves of 1.1.

### 2.2 Compared with the best possible step

In [#516](https://github.com/pasteurlabs/tesseract-core/issues/516), the ansys-shapeopt demo needed input normalization to make gradient descent work.

For each method of choosing a step :
- how far the FD gradient is from the true one, 
- how many simulation runs it costs,
- good enough for gradient descent to reach the minimum (Rosenbrock) and recover the conductivity (JAX-FEM) ?. 

Output: table method | error | `apply` calls; plot of $f(x_k) - f^\star$ against `apply` calls, one line per method.

See also: Richardson / Ridders <!-- TODO: link -->, complex step <!-- TODO: link -->.

## Chapter 3: FD for verification

### 3.1 Gradient verification with FD

`check_gradients` with each step choice from Chapter 2 (JAX-FEM), including per-input `eps` via `--eps-for` ([#713](https://github.com/pasteurlabs/tesseract-core/pull/713)).

Also: `rtol=0.1` discussion ?, as [#740](https://github.com/pasteurlabs/tesseract-core/issues/740) found on a stand-in function.

Mosaic's combines random directions, steps scaled to the input, and a sweep over `eps`.

Output: table method | correct passes | bug caught; plot of $|D_h f - \hat g| / |\hat g|$ per method, with `rtol` levels marked.

See also : noise estimation <!-- TODO: link --> ? 

### 3.2 The Taylor test

Does the convergence rate catch the 5% bug? (JAX-FEM, FEniCS). Proposed in [#740](https://github.com/pasteurlabs/tesseract-core/issues/740).

$$R(h) = |f(x + h d) - f(x) - h \langle \hat g, d \rangle|$$

$O(h^2)$ if $\hat g$ is correct / $O(h)$ otherwise.

Output: $R(h)$ on log-log axes with slope-1 and slope-2 guides, JAX-FEM against FEniCS.

### 3.3 The dot-product test

JVP against VJP, no step; catches a planted VJP bug, needs a JVP (Rosenbrock).
random directions instead of one-hot sampling ?

$$\langle u, \hat J v \rangle = \langle \hat J^\top u, v \rangle$$

Output: the mismatch for the correct and the planted-bug VJP.

## Chapter 4: Summary and recommended workflow

All methods side by side; the answer to Chapter 0. A default for each situation.

Output: the two tables sketched in #749 (`[check-gradients]`, `[ad-via-fd]`).